# 03. 特徴量エンジニアリングとモデル改善
**目的:** ベースラインXGBoostモデルに対して特徴量エンジニアリングとハイパーパラメータチューニングを重ね、
予測精度を段階的に改善する。

**このNotebookでわかること:**
- 他テーブル（CM放映・アカウント開設数）のマージによる精度向上効果
- 休日属性・過去実績ベースの特徴量が予測に寄与すること
- Optunaによるチューニングと、休日属性でモデルを分割する条件付きモデルの効果
- 対数変換・障害/振込日/確定申告期などのイベント特徴量を加えた最終モデルの精度


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data.loader import PROCESSED_DATA_DIR, save_processed
from src.features.engineering import (
    add_dow_mean_features, add_last7d_log1p_avg, add_holiday_adjacency_features,
    add_days_since_last_holiday, add_conditional_group_call_stats,
    add_tax_filing_period_flag, add_transfer_day_flag, add_incident_flag, add_spike_flag,
)
from src.models.forecasting import (
    run_xgboost_pipeline, evaluate_forecast, get_train_subset, predict_with_local_models,
    tune_xgboost, save_model, GROUP_COLS, DEFAULT_DROP_COLS, RANDOM_STATE,
)
from xgboost import XGBRegressor

TRAIN_CUTOFF = '2020-01-01'

In [ ]:
regi_call_df = pd.read_csv(PROCESSED_DATA_DIR / 'regi_call.csv', parse_dates=['cdr_date'])
regi_acc_df = pd.read_csv(PROCESSED_DATA_DIR / 'regi_acc.csv', parse_dates=['cdr_date'])
cm_df = pd.read_csv(PROCESSED_DATA_DIR / 'cm.csv', parse_dates=['cdr_date'])
gt_df = pd.read_csv(PROCESSED_DATA_DIR / 'gt.csv', parse_dates=['cdr_date'])
calender_df = regi_call_df[['cdr_date', 'holiday_flag']].drop_duplicates()

train = regi_call_df[regi_call_df['cdr_date'] < TRAIN_CUTOFF].copy()
test = regi_call_df[regi_call_df['cdr_date'] >= TRAIN_CUTOFF].copy()
for df in (train, test):
    df['week'] = df['cdr_date'].dt.isocalendar().week
    df['month'] = df['cdr_date'].dt.month

train['istrain'] = 1
test['istrain'] = 0
combined_df = pd.concat([train, test], ignore_index=True)
combined_df.shape

## 他のテーブル情報のマージ

In [ ]:
combined_df = pd.merge(combined_df, cm_df, on='cdr_date', how='left')
combined_df = pd.merge(combined_df, regi_acc_df[['cdr_date', 'acc_get_cnt']], on='cdr_date', how='left')

train = combined_df[combined_df['istrain'] == 1].drop(columns='istrain').copy()
test = combined_df[combined_df['istrain'] == 0].drop(columns='istrain').copy()
run_xgboost_pipeline(train, test)

CM放映情報とアカウント開設数をマージしただけでベースラインより精度が改善する。

## 休日情報を活用した特徴量

In [ ]:
combined_df = combined_df.sort_values('cdr_date').reset_index(drop=True)
combined_df = add_holiday_adjacency_features(combined_df)

train = combined_df[combined_df['istrain'] == 1].drop(columns='istrain').copy()
test = combined_df[combined_df['istrain'] == 0].drop(columns='istrain').copy()
run_xgboost_pipeline(train, test)

## 過去のデータを活用した特徴量

In [ ]:
# call_numをマスクしたtestデータを作成（リーク防止）
test_masked = test.copy()
test_masked['call_num'] = np.nan
combined_df_fe = pd.concat([train, test_masked], axis=0).sort_values('cdr_date').reset_index(drop=True)

#### 曜日別の入電数平均（営業日のみ）

In [ ]:
combined_df_fe = add_dow_mean_features(combined_df_fe, regi_call_df)
combined_df_fe.columns

#### アカウント開設数のラグ・移動平均特徴量
call_numを直接使うとtestデータでNaNが多くなるため、相関の強いアカウント開設数から特徴量を作る。

In [ ]:
combined_df_fe['acc_cnt_lag1'] = combined_df_fe['acc_get_cnt'].shift(1)
combined_df_fe['acc_cnt_lag7'] = combined_df_fe['acc_get_cnt'].shift(7)
combined_df_fe['acc_cnt_roll7'] = combined_df_fe['acc_get_cnt'].rolling(window=7).mean()

#### 予測対象日を含む過去7日間のアカウント開設数の対数平均

In [ ]:
combined_df_fe = add_last7d_log1p_avg(combined_df_fe)

def rebuild_train_test_fe(combined_df_fe, test):
    test_fe = combined_df_fe[combined_df_fe['cdr_date'] >= TRAIN_CUTOFF].copy()
    train_fe = combined_df_fe[combined_df_fe['cdr_date'] < TRAIN_CUTOFF].copy()
    test_fe = test_fe.drop(columns=['call_num']).merge(test[['cdr_date', 'call_num']], on='cdr_date', how='left')
    return train_fe, test_fe

train_fe, test_fe = rebuild_train_test_fe(combined_df_fe, test)
run_xgboost_pipeline(train_fe, test_fe)

## 異常値（突発的な入電数増加）を吸収するための特徴量

In [ ]:
combined_df_fe = add_spike_flag(combined_df_fe, regi_acc_df['acc_get_cnt'], 'acc_get_cnt', quantile=0.9)
train_fe, test_fe = rebuild_train_test_fe(combined_df_fe, test)
run_xgboost_pipeline(train_fe, test_fe)

段階的な特徴量追加により、ベースラインから着実に誤差が縮小している。次はハイパーパラメータのチューニングに進む。

## モデルの改善

### Optunaによるハイパラのチューニング

In [ ]:
X_train = train_fe.drop(columns=DEFAULT_DROP_COLS + ['cdr_date', 'call_num'], errors='ignore')
y_train = train_fe['call_num']
X_test = test_fe.drop(columns=DEFAULT_DROP_COLS + ['cdr_date', 'call_num'], errors='ignore')
y_test = test_fe['call_num']

best_params = tune_xgboost(X_train, y_train, X_test, y_test, n_trials=100)

In [ ]:
model = XGBRegressor(**best_params, random_state=RANDOM_STATE)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
evaluate_forecast(y_test, y_pred)

## 条件付きモデル分割
予測対象日の休日属性（曜日・祝日前後など）が近いデータだけで学習したローカルモデルと、
全体で学習したグローバルモデルをデータ件数に応じて使い分ける。

In [ ]:
global_model = XGBRegressor(**best_params, random_state=RANDOM_STATE)
global_model.fit(X_train, y_train)

train_attr = train_fe[GROUP_COLS].reset_index(drop=True)
test_attr = test_fe[GROUP_COLS].reset_index(drop=True)

y_preds = predict_with_local_models(global_model, X_train, y_train, X_test, train_attr, test_attr)
evaluate_forecast(y_test, y_preds)

In [ ]:
dow_map = {1: '月', 2: '火', 3: '水', 4: '木', 5: '金', 6: '土', 7: '日'}
segment_key = (
    train_attr['holiday_flag'].astype(int).astype(str)
    + train_attr['day_before_holiday_flag'].astype(int).astype(str)
    + train_attr['day_after_holiday_flag'].astype(int).astype(str)
    + '_' + train_attr['dow'].map(dow_map)
)
segment_counts = segment_key.value_counts().reset_index()
segment_counts.columns = ['segment_key', 'count']

plt.figure(figsize=(14, 6))
sns.barplot(data=segment_counts, x='segment_key', y='count', palette='viridis')
plt.xticks(rotation=90)
plt.axhline(10, color='red', linestyle='--', label='モデル学習閾値（10件）')
plt.title('属性組み合わせごとの学習データ件数')
plt.xlabel('属性セグメント（holiday / before / after / dow）')
plt.ylabel('件数')
plt.legend()
plt.tight_layout()
plt.show()

### call_numを対数変換してから学習させる
右裾の長いスパイクの影響を抑えるため、目的変数をlog1p変換してから条件付きモデルを学習する。

In [ ]:
y_train_log = np.log1p(train_fe['call_num'])

global_model_log = XGBRegressor(**best_params, random_state=RANDOM_STATE)
global_model_log.fit(X_train, y_train_log)

y_preds_log = predict_with_local_models(
    global_model_log, X_train, y_train_log, X_test, train_attr, test_attr, log_target=True,
)
evaluate_forecast(y_test, y_preds_log)

In [ ]:
plot_df = test_fe.loc[y_test.index, ['cdr_date']].copy()
plot_df['actual'] = y_test.values
plot_df['predicted'] = y_preds_log

plt.figure(figsize=(12, 5))
plt.plot(plot_df['cdr_date'], plot_df['actual'], label='実測値', marker='o')
plt.plot(plot_df['cdr_date'], plot_df['predicted'], label='予測値（XGBoost）', marker='o')
plt.title('XGBoost による入電数予測（対数変換 + 条件付きモデル）')
plt.xlabel('日付')
plt.ylabel('入電数')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 特徴量エンジニアリング②

### 連休明けの強調

In [ ]:
combined_df_fe = add_days_since_last_holiday(combined_df_fe)

### 休日属性グループごとの入電数実績を特徴量として追加
testデータのcall_numはそのまま使えないため、trainの同条件グループの平均・中央値を渡す。

In [ ]:
combined_df_fe = add_conditional_group_call_stats(combined_df_fe, TRAIN_CUTOFF)

### 確定申告期間フラグの追加
会計・経理業務に関連するイベントとして、確定申告期間（2/16〜3/15）フラグを追加する。

In [ ]:
combined_df_fe = add_tax_filing_period_flag(combined_df_fe)

### 関連決済サービスの振込日フラグの追加
毎月5/10/15/20/25日（休日なら翌営業日）に決済の入金が発生し、関連する問合せが増える傾向を捉える。

In [ ]:
combined_df_fe = add_transfer_day_flag(combined_df_fe, calender_df)

### 検索トレンドの追加

In [ ]:
expanded_rows = []
for _, row in gt_df.iterrows():
    for i in range(7):
        expanded_rows.append({'cdr_date': row['cdr_date'] + pd.Timedelta(days=i), 'search_cnt': row['search_cnt']})
gt_daily_df = pd.DataFrame(expanded_rows)
combined_df_fe = combined_df_fe.merge(gt_daily_df, on='cdr_date', how='left')

### 障害情報の追加
既知のサービス障害期間に問合せが急増する傾向を、障害フラグとして特徴量化する。

In [ ]:
combined_df_fe = add_incident_flag(combined_df_fe)

## 再度学習

In [ ]:
train_fe, test_fe = rebuild_train_test_fe(combined_df_fe, test)

X_train = train_fe.drop(columns=DEFAULT_DROP_COLS + ['cdr_date', 'call_num'], errors='ignore')
X_test = test_fe.drop(columns=DEFAULT_DROP_COLS + ['cdr_date', 'call_num'], errors='ignore')
y_train_log = np.log1p(train_fe['call_num'])
y_test = test_fe['call_num']

### グローバルモデルのチューニング

In [ ]:
best_params_final = tune_xgboost(X_train, y_train_log, X_test, y_test, n_trials=100, log_target=True)

In [ ]:
global_model_final = XGBRegressor(**best_params_final, random_state=RANDOM_STATE)
global_model_final.fit(X_train, y_train_log)

train_attr = train_fe[GROUP_COLS].reset_index(drop=True)
test_attr = test_fe[GROUP_COLS].reset_index(drop=True)

y_preds_final = predict_with_local_models(
    global_model_final, X_train, y_train_log, X_test, train_attr, test_attr, log_target=True,
)
evaluate_forecast(y_test, y_preds_final)

### 予測精度詳細

In [ ]:
plot_df = test_fe.loc[y_test.index, ['cdr_date']].copy()
plot_df['actual'] = y_test.values
plot_df['predicted'] = y_preds_final

plt.figure(figsize=(12, 5))
plt.plot(plot_df['cdr_date'], plot_df['actual'], label='実測値', marker='o')
plt.plot(plot_df['cdr_date'], plot_df['predicted'], label='予測値（XGBoost）', marker='o')
plt.title('XGBoost による入電数予測（最終モデル）')
plt.xlabel('日付')
plt.ylabel('入電数')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
error_df = test_fe[['cdr_date']].copy()
error_df['y_true'] = y_test.values
error_df['y_pred'] = y_preds_final
error_df['abs_error'] = (error_df['y_true'] - error_df['y_pred']).abs()
error_df['error_ratio'] = error_df['abs_error'] / (error_df['y_true'] + 1e-5)
error_df.sort_values('abs_error', ascending=False).head(20)

## 最終モデル・予測結果の保存
次のNotebook（効果測定）で使うために、最終モデルとテスト期間の予測結果を保存する。

In [ ]:
save_model(global_model_final, 'xgboost_final.joblib')

predictions_df = test_fe[['cdr_date', 'call_num']].copy()
predictions_df['y_pred'] = y_preds_final
save_processed(predictions_df, 'test_predictions.csv')